# 02 数据清洗与存储

> **作业**：P02 金融数据获取、管理与初步分析  
> **姓名**：李泽欣  
> **日期**：2026-04-07

---

## 概述

本 Notebook 完成数据清洗的全部流程：

| 步骤 | 内容 |
|------|------|
| 3.1 | 单表清洗：缺失值、日期格式、数据类型、重复值、离群值 |
| 3.2 | 宽表与长表转换 |
| 3.3 | 多表合并（个股 + 指数 + 宏观） |
| 进阶 | Parquet 格式存储与 CSV 对比 |

In [1]:
import os
import time
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import pyarrow.parquet as pq

PROJECT_ROOT = os.path.dirname(os.path.abspath('__file__'))
print(f'项目根目录: {PROJECT_ROOT}')

项目根目录: /Users/leezm/Desktop/金融数据/dshw-p01


## 1. 加载原始数据

In [2]:
# 股票信息
STOCK_INFO = {
    '000001': ('平安银行', '银行'),
    '601398': ('工商银行', '银行'),
    '002594': ('比亚迪',   '汽车'),
    '601633': ('长城汽车', '汽车'),
    '000002': ('万科A',    '房地产'),
    '600519': ('贵州茅台', '白酒'),
    '000858': ('五粮液',   '白酒'),
    '601857': ('中国石油', '能源'),
    '000063': ('中兴通讯', '通讯'),
    '002352': ('顺丰控股', '物流'),
}

# 加载个股数据
stock_data = {}
for code in STOCK_INFO:
    path = os.path.join(PROJECT_ROOT, f'data/stock/stock_{code}.csv')
    if os.path.exists(path):
        stock_data[code] = pd.read_csv(path)
        print(f'✓ 加载 stock_{code}.csv  shape={stock_data[code].shape}')

# 加载指数数据
index_300 = pd.read_csv(os.path.join(PROJECT_ROOT, 'data/index/index_000300.csv'))
index_500 = pd.read_csv(os.path.join(PROJECT_ROOT, 'data/index/index_000905.csv'))
print(f'\n✓ 沪深300: {index_300.shape}  中证500: {index_500.shape}')

# 加载宏观数据
cpi_df = pd.read_csv(os.path.join(PROJECT_ROOT, 'data/macro/macro_cpi.csv'))
m2_df  = pd.read_csv(os.path.join(PROJECT_ROOT, 'data/macro/macro_m2.csv'))
print(f'✓ CPI: {cpi_df.shape}  M2: {m2_df.shape}')

# 加载财务数据
finance_df = pd.read_csv(os.path.join(PROJECT_ROOT, 'data/finance/finance_ratios.csv'))
print(f'✓ 财务数据: {finance_df.shape}')

✓ 加载 stock_000001.csv  shape=(1512, 8)
✓ 加载 stock_601398.csv  shape=(1512, 8)
✓ 加载 stock_002594.csv  shape=(1512, 8)
✓ 加载 stock_601633.csv  shape=(1512, 8)
✓ 加载 stock_000002.csv  shape=(1512, 8)
✓ 加载 stock_600519.csv  shape=(1512, 8)
✓ 加载 stock_000858.csv  shape=(1512, 8)
✓ 加载 stock_601857.csv  shape=(1512, 8)
✓ 加载 stock_000063.csv  shape=(1512, 8)
✓ 加载 stock_002352.csv  shape=(1512, 8)

✓ 沪深300: (1512, 7)  中证500: (1512, 7)
✓ CPI: (477, 5)  M2: (218, 10)
✓ 财务数据: (100, 6)


---

## 2. 单表清洗（第 3.1 节要求）

对每只股票的原始数据逐项执行清洗，记录清洗前后的变化。

### 2.1 缺失值检测

本次真实运行结果显示，10 只股票的原始行情表都**没有缺失值**，因此没有生成缺失值汇总表。这个结果说明当前使用的 `baostock` 日频数据已经按交易日返回，停牌日或非交易日不会以“缺失行”的形式保留在原始表中。

因此，本步骤的结论不是“缺失值很多并需要处理”，而是：在本次样本中，原始股票行情数据质量较高，缺失值问题并不突出。后续仍保留缺失值处理代码，是为了保证流程具有可复现性和鲁棒性。


In [3]:
# 缺失值汇总
missing_summary = []
for code, df in stock_data.items():
    name = STOCK_INFO[code][0]
    total = len(df)
    for col in df.columns:
        n_miss = df[col].isna().sum()
        if n_miss > 0:
            missing_summary.append({
                '股票': f'{code} {name}',
                '列': col,
                '缺失数': n_miss,
                '缺失比例': f'{n_miss/total*100:.2f}%',
            })

if missing_summary:
    missing_df = pd.DataFrame(missing_summary)
    print('缺失值统计：')
    display(missing_df)
else:
    print('✓ 所有股票数据无缺失值！')
    print('（akshare 返回的后复权数据通常已剔除停牌日期，故无缺失）')

✓ 所有股票数据无缺失值！
（akshare 返回的后复权数据通常已剔除停牌日期，故无缺失）


### 2.2 缺失值处理

由于上一步检测结果为“所有股票数据无缺失值”，所以本次真实运行中，向前填充、成交量填 0、删除残余缺失行这些处理逻辑实际上**没有触发实质性数据变动**。代码运行完成后，输出仅显示“缺失值处理完成”，没有任何股票出现“处理前后缺失数变化”的记录。

这说明在当前样本下，缺失值处理更多是一个预防性步骤，而不是决定性清洗步骤。把这类代码保留下来仍然有意义，因为如果未来更换股票池或更换数据源，流程仍能自动应对潜在缺失问题。


In [4]:
for code in stock_data:
    df = stock_data[code]
    before = df.isna().sum().sum()
    
    # 价格列向前填充
    price_cols = ['open', 'close', 'high', 'low']
    for col in price_cols:
        if col in df.columns:
            df[col] = df[col].ffill()
    
    # 成交量/成交额填0
    for col in ['volume', 'amount']:
        if col in df.columns:
            df[col] = df[col].fillna(0)
    
    # 删除仍有缺失的行
    df = df.dropna()
    
    after = df.isna().sum().sum()
    stock_data[code] = df
    if before > 0:
        print(f'{code}: 缺失值 {before} -> {after}')

print('✓ 缺失值处理完成')

✓ 缺失值处理完成


### 2.3 日期格式统一

确保所有表的日期列统一为 `datetime64` 格式，并设为索引。这是后续合并、时间序列分析的基础。

In [5]:
for code in stock_data:
    df = stock_data[code]
    print(f'{code} 清洗前 date 类型: {df["date"].dtype}', end=' -> ')
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date').sort_index()
    stock_data[code] = df
    print(f'清洗后: datetime64, 索引范围 {df.index.min().date()} ~ {df.index.max().date()}')

# 指数数据同样处理
for name, idx_df in [('沪深300', index_300), ('中证500', index_500)]:
    date_col = [c for c in idx_df.columns if 'date' in c.lower()][0]
    idx_df[date_col] = pd.to_datetime(idx_df[date_col])
    idx_df = idx_df.set_index(date_col).sort_index()
    if name == '沪深300':
        index_300 = idx_df
    else:
        index_500 = idx_df
    print(f'{name} 日期已转为 datetime64')

print('\n✓ 日期格式统一完成')

000001 清洗前 date 类型: str -> 清洗后: datetime64, 索引范围 2020-01-02 ~ 2026-04-01
601398 清洗前 date 类型: str -> 清洗后: datetime64, 索引范围 2020-01-02 ~ 2026-04-01
002594 清洗前 date 类型: str -> 清洗后: datetime64, 索引范围 2020-01-02 ~ 2026-04-01
601633 清洗前 date 类型: str -> 清洗后: datetime64, 索引范围 2020-01-02 ~ 2026-04-01
000002 清洗前 date 类型: str -> 清洗后: datetime64, 索引范围 2020-01-02 ~ 2026-04-01
600519 清洗前 date 类型: str -> 清洗后: datetime64, 索引范围 2020-01-02 ~ 2026-04-01
000858 清洗前 date 类型: str -> 清洗后: datetime64, 索引范围 2020-01-02 ~ 2026-04-01
601857 清洗前 date 类型: str -> 清洗后: datetime64, 索引范围 2020-01-02 ~ 2026-04-01
000063 清洗前 date 类型: str -> 清洗后: datetime64, 索引范围 2020-01-02 ~ 2026-04-01
002352 清洗前 date 类型: str -> 清洗后: datetime64, 索引范围 2020-01-02 ~ 2026-04-01
沪深300 日期已转为 datetime64
中证500 日期已转为 datetime64

✓ 日期格式统一完成


### 2.4 数据类型检查

确认价格列、成交量列为数值型。如果存在字符型（可能是数据源问题），则进行转换并记录。

In [6]:
type_issues = []

for code in stock_data:
    df = stock_data[code]
    numeric_cols = ['open', 'close', 'high', 'low', 'volume', 'amount']
    for col in numeric_cols:
        if col in df.columns and not pd.api.types.is_numeric_dtype(df[col]):
            type_issues.append(f'{code}: {col} 类型为 {df[col].dtype}，需转换')
            df[col] = pd.to_numeric(df[col], errors='coerce')
    stock_data[code] = df

if type_issues:
    print('发现类型问题并已修复：')
    for issue in type_issues:
        print(f'  ⚠ {issue}')
else:
    print('✓ 所有价格和成交量列均为数值型，无需转换')

# 展示一只股票的数据类型
sample_code = list(stock_data.keys())[0]
print(f'\n示例 ({sample_code}) 数据类型：')
print(stock_data[sample_code].dtypes)

✓ 所有价格和成交量列均为数值型，无需转换

示例 (000001) 数据类型：
open      float64
high      float64
low       float64
close     float64
volume      int64
amount    float64
code        int64
dtype: object


### 2.5 重复值处理

检测并删除重复行。在日度行情中，同一股票同一日期不应有两条记录。

In [7]:
total_dup = 0
for code in stock_data:
    df = stock_data[code]
    n_dup = df.index.duplicated().sum()
    if n_dup > 0:
        print(f'{code}: 发现 {n_dup} 条重复记录，已删除')
        df = df[~df.index.duplicated(keep='first')]
        stock_data[code] = df
        total_dup += n_dup

if total_dup == 0:
    print('✓ 所有股票数据无重复行')
else:
    print(f'\n共删除 {total_dup} 条重复记录')

✓ 所有股票数据无重复行


### 2.6 离群值标注

计算日收益率，对单日涨跌幅超过 ±20% 的记录标注为 `is_extreme = True`。

**说明**：不删除离群值，仅标注。极端涨跌幅可能原因：
- 涨跌停板（A 股主板 ±10%，创业板/科创板 ±20%）
- 复权因子调整导致的计算跳跃
- 重大事件（如业绩暴雷、政策利好等）

In [8]:
extreme_summary = []

for code in stock_data:
    df = stock_data[code]
    # 计算日对数收益率
    df['return'] = np.log(df['close'] / df['close'].shift(1))
    # 标注极端值
    df['is_extreme'] = df['return'].abs() > 0.20
    n_extreme = df['is_extreme'].sum()
    stock_data[code] = df
    
    name, industry = STOCK_INFO[code]
    extreme_summary.append({
        '股票': f'{code} {name}',
        '行业': industry,
        '总交易日': len(df),
        '极端值数量': n_extreme,
        '占比': f'{n_extreme/len(df)*100:.2f}%',
    })

extreme_df = pd.DataFrame(extreme_summary)
print('离群值标注统计（|日收益率| > 20%）：')
display(extreme_df)

# 展示部分极端值记录
for code in stock_data:
    df = stock_data[code]
    extremes = df[df['is_extreme']]
    if len(extremes) > 0:
        name = STOCK_INFO[code][0]
        print(f'\n{code} {name} 极端值记录：')
        display(extremes[['close', 'return', 'is_extreme']].head(5))

离群值标注统计（|日收益率| > 20%）：


,股票,行业,总交易日,极端值数量,占比
0,000001 平安银行,银行,1512,0,0.00%
1,601398 工商银行,银行,1512,0,0.00%
2,002594 比亚迪,汽车,1512,0,0.00%
3,601633 长城汽车,汽车,1512,0,0.00%
4,000002 万科A,房地产,1512,0,0.00%
5,600519 贵州茅台,白酒,1512,0,0.00%
6,000858 五粮液,白酒,1512,0,0.00%
7,601857 中国石油,能源,1512,0,0.00%
8,000063 中兴通讯,通讯,1512,0,0.00%
9,002352 顺丰控股,物流,1512,0,0.00%


**离群值分析**：

本次 10 只股票在 **1512 个交易日** 的样本中，按“日对数收益率绝对值大于 20%”的标准统计，**极端值数量全部为 0**。这与 A 股主板和大盘蓝筹样本的交易制度是一致的，说明当前使用的后复权收盘价序列没有出现异常跳点，也没有显著受到复权处理带来的极端收益率干扰。

因此，在本次样本下，`is_extreme` 更像是一个风险监控字段，而不是需要重点处理的问题字段。这个结果也意味着后续描述统计和 CAPM 估计受到异常值污染的风险相对较低。

---


## 3. 宽表与长表转换（第 3.2 节要求）

### 宽表（Wide format）
将 10 只股票的收盘价合并为宽表：日期为索引，每列为一只股票。

**宽表适用场景**：计算相关系数矩阵、绘制多股票走势图、计算协方差矩阵等。

### 长表（Long format）
使用 `pd.melt` 转换回长表，字段为 `date, code, close`。

**长表适用场景**：分组统计（groupby）、数据合并（merge）、用 seaborn 等工具绑定变量绘图。

In [9]:
# --- 构建宽表 ---
close_wide = pd.DataFrame()
for code, df in stock_data.items():
    close_wide[code] = df['close']

print(f'宽表形状: {close_wide.shape}  （行=交易日, 列=股票）')
print('\n宽表前5行：')
display(close_wide.head())

# --- 宽表转长表 ---
close_long = close_wide.reset_index().melt(
    id_vars='date',
    var_name='code',
    value_name='close'
)
close_long = close_long.sort_values(['code', 'date']).reset_index(drop=True)

print(f'\n长表形状: {close_long.shape}  （行=股票×交易日, 列=3）')
print('\n长表前10行：')
display(close_long.head(10))

宽表形状: (1512, 10)  （行=交易日, 列=股票）

宽表前5行：


,000001,601398,002594,601633,000002,600519,000858,601857,000063,002352
date,,,,,,,,,,
2020-01-02,1989.482980,10.605060,49.092745,33.300072,3590.897899,7334.976900,1955.521403,7.868847,447.732791,129.552806
2020-01-03,2026.041351,10.640588,48.960254,33.191484,3534.652262,7001.073173,1932.868861,7.976088,461.625769,128.550396
2020-01-06,2013.069026,10.605060,49.204852,32.431374,3475.098059,6997.373229,1912.881324,8.351433,463.520266,127.375157
2020-01-07,2022.503444,10.676116,48.970446,33.119093,3502.669449,7104.736519,1915.398273,8.190571,462.257268,130.313254
2020-01-08,1964.717632,10.498477,48.185696,32.503766,3493.846604,7063.258198,1908.291593,8.351433,451.016586,128.757791



长表形状: (15120, 3)  （行=股票×交易日, 列=3）

长表前10行：


,date,code,close
0,2020-01-02,000001,1989.482980
1,2020-01-03,000001,2026.041351
2,2020-01-06,000001,2013.069026
3,2020-01-07,000001,2022.503444
4,2020-01-08,000001,1964.717632
5,2020-01-09,000001,1980.048562
6,2020-01-10,000001,1968.255539
7,2020-01-13,000001,2003.634608
8,2020-01-14,000001,1976.510655
9,2020-01-15,000001,1948.207400


**宽表 vs 长表总结**：

本次真实转换结果显示，收盘价宽表的形状为 **(1512, 10)**，表示 1512 个交易日对应 10 只股票；转换后的长表形状为 **(15120, 3)**，恰好等于“交易日数 × 股票数”。这说明宽表与长表之间的转换是成功且一致的，没有出现行数丢失。

从用途看，宽表非常适合本作业后续的相关系数计算、收益率矩阵构造和多资产折线图绘制；长表则更适合按股票分组汇总、与行业信息合并，以及使用 `seaborn` 做带 `hue` 或分面的可视化。因此，本次作业中两种格式都不是“谁替代谁”，而是分别服务于不同分析任务。

---


## 4. 多表合并（第 3.3 节要求）

### 4.1 个股 + 指数合并

将个股日度数据与沪深 300 指数按日期做 `left join`。

In [10]:
# 合并所有个股为一个长表
all_stocks = []
for code, df in stock_data.items():
    temp = df.copy()
    temp['code'] = code
    temp['name'] = STOCK_INFO[code][0]
    temp['industry'] = STOCK_INFO[code][1]
    all_stocks.append(temp)

stock_all = pd.concat(all_stocks)
print(f'合并前：个股长表 {stock_all.shape[0]} 行')

# 准备指数数据
idx_close_col = [c for c in index_300.columns if 'close' in c.lower()]
if idx_close_col:
    index_300_close = index_300[[idx_close_col[0]]].rename(columns={idx_close_col[0]: 'index_300_close'})
else:
    # 尝试其他可能的列名
    print('沪深300列名:', index_300.columns.tolist())
    index_300_close = index_300.iloc[:, :1]
    index_300_close.columns = ['index_300_close']

# 合并
stock_with_index = stock_all.join(index_300_close, how='left')
print(f'合并后：{stock_with_index.shape[0]} 行')
print(f'行数变化说明：left join 不改变左表行数，仅添加指数列。某些日期指数无数据时该列为 NaN。')
print(f'\n指数列缺失值: {stock_with_index["index_300_close"].isna().sum()} 个')

合并前：个股长表 15120 行
合并后：15120 行
行数变化说明：left join 不改变左表行数，仅添加指数列。某些日期指数无数据时该列为 NaN。

指数列缺失值: 0 个


### 4.2 月度宏观数据与日度数据合并

本次运行中，个股与沪深 300 的 `left join` 是成功的：合并前后行数都为 **15120**，且指数列缺失值为 **0**。这说明股票日度数据与指数日度数据在日期维度上能够稳定对齐。

修复 CPI 字段识别逻辑后，月度 CPI 数据也已经成功映射到日度综合表。当前 `cpi` 列共有 **13730** 行非缺失值，仅有 **1390** 行缺失，这些缺失主要来自 **2025-09 至 2026-04** 期间股票样本超出了当前 CPI 数据的可用发布时间范围。

因此，当前 `combined_data.csv` 已经可以用于后续基于综合表的宏观联动分析。对于尾部缺失月份，解释应当是“宏观数据尚未发布”，而不是“合并逻辑失败”。


In [11]:
# 处理 CPI 数据 —— 按当前数据的真实字段明确指定
print('CPI 数据列名:', cpi_df.columns.tolist())
print('CPI 前5行:')
display(cpi_df.head())

cpi_clean = cpi_df.copy()
date_col = '日期' if '日期' in cpi_clean.columns else cpi_clean.columns[0]
val_col = '今值' if '今值' in cpi_clean.columns else cpi_clean.columns[2]

print(f'\n使用日期列: {date_col}, 数值列: {val_col}')

cpi_clean['date'] = pd.to_datetime(cpi_clean[date_col], errors='coerce')
cpi_clean['cpi'] = pd.to_numeric(cpi_clean[val_col], errors='coerce')
cpi_clean = cpi_clean.dropna(subset=['date', 'cpi'])
cpi_clean['year_month'] = cpi_clean['date'].dt.to_period('M')
cpi_monthly = cpi_clean[['year_month', 'cpi']].drop_duplicates('year_month').sort_values('year_month')
print(f'CPI 月度数据: {len(cpi_monthly)} 条')
print(f'最新可用月份: {cpi_monthly["year_month"].max()}')


CPI 数据列名: ['商品', '日期', '今值', '预测值', '前值']
CPI 前5行:


,商品,日期,今值,预测值,前值
0,中国CPI年率报告,1986-02-01,7.1,NaN,NaN
1,中国CPI年率报告,1986-03-01,7.1,NaN,7.1
2,中国CPI年率报告,1986-04-01,7.1,NaN,7.1
3,中国CPI年率报告,1986-05-01,7.1,NaN,7.1
4,中国CPI年率报告,1986-06-01,7.1,NaN,7.1



使用日期列: 日期, 数值列: 今值
CPI 月度数据: 475 条
最新可用月份: 2025-08


In [12]:
# 将月度 CPI 映射到日度数据
stock_with_index_reset = stock_with_index.reset_index()
stock_with_index_reset['year_month'] = pd.to_datetime(stock_with_index_reset['date']).dt.to_period('M')

before_rows = len(stock_with_index_reset)
combined = stock_with_index_reset.merge(cpi_monthly, on='year_month', how='left')
after_rows = len(combined)

print(f'合并前行数: {before_rows}')
print(f'合并后行数: {after_rows}')
print(f'行数变化: {after_rows - before_rows}（left join 理论上不增加行数；若增加可能是 CPI 有重复月份）')
print(f'CPI 列非缺失值: {combined["cpi"].notna().sum()} 个')
print(f'CPI 列缺失值: {combined["cpi"].isna().sum()} 个（主要来自超出当前 CPI 数据发布时间范围的月份）')


合并前行数: 15120
合并后行数: 15120
行数变化: 0（left join 理论上不增加行数；若增加可能是 CPI 有重复月份）
CPI 列非缺失值: 13730 个
CPI 列缺失值: 1390 个（主要来自超出当前 CPI 数据发布时间范围的月份）


---

## 5. 保存清洗后的数据

### 5.1 方式 A：CSV 格式（必做）

In [13]:
# 保存清洗后的个股数据（合并为一个文件）
stock_clean = pd.concat([df.assign(code=code) for code, df in stock_data.items()])
stock_clean = stock_clean.reset_index()

csv_path = os.path.join(PROJECT_ROOT, 'data/clean/stock_clean.csv')
stock_clean.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f'✓ 清洗后个股数据已保存: {csv_path}  shape={stock_clean.shape}')

# 保存合并后的综合数据
combined_path = os.path.join(PROJECT_ROOT, 'data/combined/combined_data.csv')
combined.to_csv(combined_path, index=False, encoding='utf-8-sig')
print(f'✓ 综合数据已保存: {combined_path}  shape={combined.shape}')

✓ 清洗后个股数据已保存: /Users/leezm/Desktop/金融数据/dshw-p01/data/clean/stock_clean.csv  shape=(15120, 10)
✓ 综合数据已保存: /Users/leezm/Desktop/金融数据/dshw-p01/data/combined/combined_data.csv  shape=(15120, 15)


### 5.2 方式 B：Parquet 格式（进阶）

将清洗后的数据额外保存为 Parquet 格式，并与 CSV 做对比。

In [14]:
# 保存 Parquet
parquet_path = os.path.join(PROJECT_ROOT, 'data/clean/stock_clean.parquet')
stock_clean.to_parquet(parquet_path, index=False)
print(f'✓ Parquet 格式已保存: {parquet_path}')

✓ Parquet 格式已保存: /Users/leezm/Desktop/金融数据/dshw-p01/data/clean/stock_clean.parquet


In [15]:
# --- Parquet 特性演示 ---

# 1. 列式读取（只加载需要的列）
df_partial = pd.read_parquet(parquet_path, columns=['date', 'code', 'close'])
print('列式读取（仅 date, code, close）：')
display(df_partial.head())

# 2. 查看 Schema（类型契约）
schema = pq.read_schema(parquet_path)
print('\nParquet Schema：')
print(schema)

列式读取（仅 date, code, close）：


,date,code,close
0,2020-01-02,000001,1989.482980
1,2020-01-03,000001,2026.041351
2,2020-01-06,000001,2013.069026
3,2020-01-07,000001,2022.503444
4,2020-01-08,000001,1964.717632



Parquet Schema：
date: timestamp[us]
open: double
high: double
low: double
close: double
volume: int64
amount: double
code: large_string
return: double
is_extreme: bool
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1200


In [16]:
# 3. CSV vs Parquet 对比：读取速度与文件体积
import time, os

csv_file = os.path.join(PROJECT_ROOT, 'data/clean/stock_clean.csv')
pq_file  = os.path.join(PROJECT_ROOT, 'data/clean/stock_clean.parquet')

# CSV 读取
t0 = time.time()
pd.read_csv(csv_file)
csv_time = time.time() - t0
csv_size = os.path.getsize(csv_file) / 1024

# Parquet 读取
t0 = time.time()
pd.read_parquet(pq_file)
pq_time = time.time() - t0
pq_size = os.path.getsize(pq_file) / 1024

print(f'CSV     读取耗时: {csv_time:.3f}s  文件大小: {csv_size:.1f} KB')
print(f'Parquet 读取耗时: {pq_time:.3f}s  文件大小: {pq_size:.1f} KB')
print(f'\n体积比: Parquet / CSV = {pq_size/csv_size:.1%}')
print(f'速度比: Parquet / CSV = {pq_time/csv_time:.1%}')

CSV     读取耗时: 0.006s  文件大小: 1734.4 KB
Parquet 读取耗时: 0.002s  文件大小: 885.7 KB

体积比: Parquet / CSV = 51.1%
速度比: Parquet / CSV = 23.8%


### Parquet vs CSV 对比总结

本次真实运行结果显示，`stock_clean.csv` 的读取耗时约为 **0.006 秒**，文件大小约 **1734.4 KB**；`stock_clean.parquet` 的读取耗时约为 **0.001 秒**，文件大小约 **885.7 KB**。换句话说，Parquet 文件体积约为 CSV 的 **51.1%**，读取速度约为 CSV 的 **21.0%**。

这个差异已经不是“几乎看不出来”，而是在当前约 1.5 万行的数据规模下就已经具有可感知优势。虽然绝对耗时都很短，但 Parquet 在体积压缩和读取效率上的优势已经体现出来；如果未来扩展到更多股票、更长时间区间或更高频数据，这种差异还会进一步放大。


## 6. 清洗结果总览

基于本次真实运行结果，数据清洗阶段的主要结论如下：

- 原始股票数据整体质量较高，**无缺失值、无重复值、无超过 20% 的极端收益率**
- 日期字段已统一转换为 `datetime64`，并成功建立时间索引
- 宽表与长表转换正确，宽表 **(1512, 10)**，长表 **(15120, 3)**
- 个股与指数合并成功，CPI 月度数据也已成功映射到综合表，其中 `cpi` 列 **13730** 行非缺失、**1390** 行缺失
- 清洗后个股数据已保存为 CSV 和 Parquet 两种格式，其中 Parquet 在本次样本下已经表现出明显的体积和速度优势

因此，本 Notebook 目前已经完成了作业要求中的单表清洗、格式转换、多表合并以及 CSV/Parquet 存储任务。当前 `cpi` 列中剩余缺失值反映的是宏观数据发布时间覆盖范围，而不是代码错误。


In [17]:
print('=' * 60)
print('数据清洗完成！结果总览：')
print('=' * 60)
print(f'\n清洗后个股数据: {stock_clean.shape}')
print(f'综合合并数据:   {combined.shape}')
print(f'\n文件输出:')
for f in ['data/clean/stock_clean.csv', 'data/clean/stock_clean.parquet', 'data/combined/combined_data.csv']:
    fp = os.path.join(PROJECT_ROOT, f)
    if os.path.exists(fp):
        size = os.path.getsize(fp) / 1024
        print(f'  ✓ {f}  ({size:.1f} KB)')

print('\n下一步：运行 03_analysis.ipynb 进行描述性统计与回归分析')

数据清洗完成！结果总览：

清洗后个股数据: (15120, 10)
综合合并数据:   (15120, 15)

文件输出:
  ✓ data/clean/stock_clean.csv  (1734.4 KB)
  ✓ data/clean/stock_clean.parquet  (885.7 KB)
  ✓ data/combined/combined_data.csv  (2339.1 KB)

下一步：运行 03_analysis.ipynb 进行描述性统计与回归分析
